# Live Plot - Mode B with a callback

Mode B runs the simulation on the calling thread and calls a Python
callback at a fixed interval, with the GIL released in between.  Here the
callback redraws the vacancy depth profile so you can watch the statistics
converge.

This uses the inline backend so it runs anywhere (including headless).
For smoother in-place updates in JupyterLab install ipympl and switch the
first line to `%matplotlib widget`.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from IPython import display

import opentrim

In [ ]:
config = opentrim.Config()
config.IonBeam.ion = opentrim.Element('He')
config.IonBeam.energy_distribution.center = 2e6

mat = opentrim.Material(); mat.id = 'Fe'; mat.density = 7.874
atom = opentrim.Atom(); atom.element = opentrim.Element('Fe')
atom.X = 1.0; atom.Ed = 40.0; atom.El = 3.0; atom.Es = 4.28; atom.Er = 40.0; atom.Rc = 0.946
mat.composition.append(atom)
config.Target.materials.append(mat)

region = opentrim.Region(); region.id = 'slab'; region.material_id = 'Fe'
region.origin = [0, 0, 0]; region.size = [100, 100, 100]
config.Target.regions.append(region)

config.Target.cell_count = [200, 1, 1]
config.Target.size = [100.0, 100.0, 100.0]
config.Simulation.electronic_stopping = opentrim.Stopping.SRIM13
config.Simulation.simulation_type = opentrim.SimulationType.FullCascade
config.Output.outfilename = 'He_2MeV_Fe_live'
config.Run.max_no_ions = 20000
config.Run.threads = 4
config.validate()

In [ ]:
sim = opentrim.Driver()
sim.init(config)

info = opentrim.Info(sim)
x_edges = info['target']['grid']['X']
x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])

fig, ax = plt.subplots(figsize=(8, 4))

def live_update():
    v, dv = info['tally']['damage_events']['Vacancies']
    v_depth = v.sum(axis=0)[:, 0, 0]
    ax.clear()
    ax.plot(x_centers, v_depth)
    ax.set_xlabel('Depth (nm)')
    ax.set_ylabel('Vacancies per ion')
    ax.set_title(f'N = {sim.ion_count():,} ions')
    fig.canvas.draw()
    display.clear_output(wait=True)
    display.display(fig)

In [ ]:
# Mode B: blocking run, callback every 250 ms.  Ctrl+C raises
# KeyboardInterrupt at the next callback and aborts the run.
sim.run(live_update, interval_ms=250)
print('final ions:', sim.ion_count())